# Week 7 — Retrieval-Augmented Generation (RAG) System
### Celebal Summer Internship — Data Science Track

**Goal:** Build a simple end-to-end RAG pipeline that:
1. Ingests documents (PDF / raw text / Hugging Face dataset)
2. Chunks the text
3. Embeds the chunks with a pretrained embedding model
4. Stores embeddings in a vector database
5. Embeds the user's query and retrieves the most relevant chunks
6. Feeds the query + retrieved context into an LLM to generate a grounded answer
7. Experiments with an optimization (hybrid keyword+vector search)

**Dataset used:** [`rag-datasets/rag-mini-wikipedia`](https://huggingface.co/datasets/rag-datasets/rag-mini-wikipedia) — a small Hugging Face dataset built specifically for RAG tutorials. It ships with:
- a `text-corpus` split (short Wikipedia passages — our "documents")
- a `question-answer` split (real question/ground-truth-answer pairs we can use to validate retrieval quality)

**Embedding model:** `sentence-transformers/all-MiniLM-L6-v2` (384-dim, fast, free, runs on CPU)

**Vector store:** FAISS (`IndexFlatIP` — exact cosine similarity search)

**Generator LLM:** `google/flan-t5-base` (free, no API key, runs in Colab)

> Run every cell top to bottom in **Google Colab** (Runtime → Run all). Cells marked `# YOUR CODE HERE` are for you to fill in — everything else is prewritten scaffolding.


In [17]:
!pip install -q datasets sentence-transformers faiss-cpu transformers rank_bm25 accelerate


In [18]:
import numpy as np
import pandas as pd
import textwrap
import time

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import faiss
from transformers import pipeline
from rank_bm25 import BM25Okapi

import warnings
warnings.filterwarnings("ignore")

print("Libraries imported successfully.")


Libraries imported successfully.


## 1. Document Ingestion Module

A real RAG system needs to accept documents from **multiple sources**: a PDF a user uploads, a plain `.txt` file, or a ready-made Hugging Face dataset. Below we build one ingestion function that handles all three, then use the Hugging Face path for this notebook (`rag-mini-wikipedia`).


In [19]:
def ingest_documents(source_type: str, source_path: str = None, hf_dataset_name: str = None, hf_config: str = None, hf_split: str = "train", text_column: str = "passage"):

    documents = []

    if source_type == "pdf":
        import PyPDF2
        with open(source_path, "rb") as f:
            reader = PyPDF2.PdfReader(f)
            for page in reader.pages:
                text = page.extract_text()
                if text:
                    documents.append(text)

    elif source_type == "txt":
        with open(source_path, "r", encoding="utf-8") as f:
            documents.append(f.read())

    elif source_type == "huggingface":
        ds = load_dataset(hf_dataset_name, hf_config, split=hf_split)
        documents = [row[text_column] for row in ds if row[text_column] and row[text_column].strip()]

    else:
        raise ValueError("source_type must be one of: 'pdf', 'txt', 'huggingface'")

    print(f"Ingested {len(documents)} document(s)/passage(s) from source_type='{source_type}'")
    return documents


# Ingest the RAG-mini-Wikipedia text corpus (our "custom document" collection)
raw_documents = ingest_documents(
    source_type="huggingface",
    hf_dataset_name="rag-datasets/rag-mini-wikipedia",
    hf_config="text-corpus",
    hf_split="passages",
    text_column="passage"
)

print("\nSample document:\n")
print(textwrap.fill(raw_documents[0], width=100))


Ingested 3200 document(s)/passage(s) from source_type='huggingface'

Sample document:

Uruguay (official full name in  ; pron.  , Eastern Republic of  Uruguay) is a country located in the
southeastern part of South America.  It is home to 3.3 million people, of which 1.7 million live in
the capital Montevideo and its metropolitan area.


## 2. Chunking Module

Language models and embedding models have limited context windows, and retrieval works best on small, semantically focused pieces of text. We chunk each document using a **sliding-window (character-based, with overlap)** strategy: this keeps chunks a consistent size while the overlap prevents cutting a sentence/idea in half at chunk boundaries.


In [20]:
def clean_text(text: str) -> str:
    """Basic text cleaning: collapse whitespace, strip."""
    return " ".join(text.split()).strip()


def chunk_text(text: str, chunk_size: int = 300, chunk_overlap: int = 50):
    """
    Splits `text` into overlapping chunks of ~chunk_size characters.

    chunk_size    : target number of characters per chunk
    chunk_overlap : number of overlapping characters between consecutive chunks
                    (preserves context continuity across chunk boundaries)

    Returns: list[str] chunks
    """
    text = clean_text(text)
    if len(text) <= chunk_size:
        return [text]

    chunks = []
    start = 0
    step = chunk_size - chunk_overlap
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += step
    return chunks


CHUNK_SIZE = 300
CHUNK_OVERLAP = 50

all_chunks = []
chunk_source_ids = []   # keeps track of which original document a chunk came from

for doc_id, doc in enumerate(raw_documents):
    doc_chunks = chunk_text(doc, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    all_chunks.extend(doc_chunks)
    chunk_source_ids.extend([doc_id] * len(doc_chunks))

print(f"Total documents: {len(raw_documents)}")
print(f"Total chunks created: {len(all_chunks)}")
print(f"Chunk size: {CHUNK_SIZE} chars | Overlap: {CHUNK_OVERLAP} chars")
print("\nExample chunk:\n")
print(textwrap.fill(all_chunks[0], width=100))


Total documents: 3200
Total chunks created: 6581
Chunk size: 300 chars | Overlap: 50 chars

Example chunk:

Uruguay (official full name in ; pron. , Eastern Republic of Uruguay) is a country located in the
southeastern part of South America. It is home to 3.3 million people, of which 1.7 million live in
the capital Montevideo and its metropolitan area.


## 3. Embedding Module

Each chunk is mapped to a dense vector using a pretrained sentence-embedding model. Chunks that are semantically similar end up close together in vector space — this is what makes similarity search possible.


In [21]:
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print("Encoding chunks into embeddings (this may take a minute)...")
start = time.time()
chunk_embeddings = embedding_model.encode(
    all_chunks,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True   # pre-normalize so inner product == cosine similarity
)
print(f"Done in {time.time() - start:.1f}s")

EMBEDDING_DIM = chunk_embeddings.shape[1]
print(f"\nEmbedding matrix shape: {chunk_embeddings.shape}")
print(f"Embedding dimension: {EMBEDDING_DIM}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Encoding chunks into embeddings (this may take a minute)...


Batches:   0%|          | 0/206 [00:00<?, ?it/s]

Done in 218.6s

Embedding matrix shape: (6581, 384)
Embedding dimension: 384


## 4. Vector Database

We use **FAISS** (`IndexFlatIP`) to store all chunk embeddings and perform fast similarity search. Since embeddings are normalized, inner product search here is equivalent to cosine similarity search.


In [22]:
vector_index = faiss.IndexFlatIP(EMBEDDING_DIM)
vector_index.add(chunk_embeddings.astype("float32"))

print(f"Vector store initialized: FAISS IndexFlatIP")
print(f"Number of vectors stored: {vector_index.ntotal}")


Vector store initialized: FAISS IndexFlatIP
Number of vectors stored: 6581


## 5. Query Embedding + Retrieval Module

When a user asks a question, we embed the query with the **same** embedding model, then search the vector store for the top-k most similar chunks.


In [23]:
def embed_query(query: str) -> np.ndarray:
    """Embeds a user query using the same embedding model as the document chunks."""
    return embedding_model.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype("float32")


def retrieve(query: str, k: int = 5):
    """
    Retrieval module: embeds the query and returns the top-k most relevant chunks.

    Returns: list of dicts -> {'chunk': str, 'score': float, 'source_doc_id': int}
    """
    query_vec = embed_query(query)
    scores, indices = vector_index.search(query_vec, k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        results.append({
            "chunk": all_chunks[idx],
            "score": float(score),
            "source_doc_id": chunk_source_ids[idx]
        })
    return results


# Quick sanity check
sample_results = retrieve("What is the capital of France?", k=3)
for r in sample_results:
    print(f"[score={r['score']:.3f}] {textwrap.fill(r['chunk'], width=100)}\n")


[score=0.468] tions and La Francophonie (French-Speaking Countries).

[score=0.426] French explorer Samuel de Champlain arrived in 1603 and established the first permanent European
settlements at Port Royal in 1605 and Quebec City in 1608. These would become respectively the
capitals of Acadia and Canada. Among French colonists of New France, Canadiens extensively settled
the St. L

[score=0.409] Montevideo, Uruguay's capital.



## 6. Hybrid Search — Optimization Experiment (keyword + vector)

Pure vector search can miss exact keyword matches (e.g. rare proper nouns, IDs, numbers). We build a **BM25 keyword index** alongside the vector index, then fuse both scores for a hybrid retriever — this is one of the optional optimizations listed in the assignment.


In [24]:
tokenized_chunks = [c.lower().split() for c in all_chunks]
bm25 = BM25Okapi(tokenized_chunks)


def hybrid_retrieve(query: str, k: int = 5, alpha: float = 0.5):
    """
    Hybrid retrieval: combines normalized BM25 (keyword) scores with
    normalized vector similarity scores.

    alpha : weight given to the vector score (1-alpha goes to BM25)
    """
    # Vector similarity over ALL chunks
    query_vec = embed_query(query)
    vector_scores = (chunk_embeddings @ query_vec.T).flatten()

    # BM25 keyword scores over ALL chunks
    bm25_scores = np.array(bm25.get_scores(query.lower().split()))

    # Min-max normalize both score arrays to [0, 1] so they're comparable
    def normalize(arr):
        if arr.max() == arr.min():
            return np.zeros_like(arr)
        return (arr - arr.min()) / (arr.max() - arr.min())

    v_norm = normalize(vector_scores)
    b_norm = normalize(bm25_scores)

    hybrid_scores = alpha * v_norm + (1 - alpha) * b_norm
    top_k_idx = np.argsort(hybrid_scores)[::-1][:k]

    results = []
    for idx in top_k_idx:
        results.append({
            "chunk": all_chunks[idx],
            "score": float(hybrid_scores[idx]),
            "source_doc_id": chunk_source_ids[idx]
        })
    return results


# Compare vector-only vs hybrid retrieval on the same query
test_query = "Notre Dame Fighting Irish football"
print("--- Vector-only retrieval ---")
for r in retrieve(test_query, k=3):
    print(f"[{r['score']:.3f}] {r['chunk'][:100]}...")

print("\n--- Hybrid (BM25 + vector) retrieval ---")
for r in hybrid_retrieve(test_query, k=3):
    print(f"[{r['score']:.3f}] {r['chunk'][:100]}...")


--- Vector-only retrieval ---
[0.377] Play fight...
[0.356] Though maybe not the force they once were, the Romanian national rugby team has so far competed at e...
[0.334] Irish Americans were powerful in the Democratic party and opposed going to war alongside their enemy...

--- Hybrid (BM25 + vector) retrieval ---
[0.878] rmoil, with German voters outraged at their wartime harassment, and Irish voters angry at his failur...
[0.832] n attacking Imperial Russian and Imperial German fighting forces, causing the two fighting armies to...
[0.738] Ford as a University of Michigan football player, 1933...


## 7. Generation Module — connecting retrieval to the LLM

We build a prompt template that injects the retrieved context chunks + the original query, then pass it to a free, local Hugging Face LLM (`google/flan-t5-base`) to produce a grounded answer.


In [25]:
# NOTE: newer releases of `transformers` (v5+) removed the 'text2text-generation'
# pipeline() task from the registry, so we load the seq2seq model + tokenizer directly
# instead of relying on pipeline(). This is version-proof and works everywhere.

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

GENERATOR_MODEL_NAME = "google/flan-t5-base"

gen_tokenizer = AutoTokenizer.from_pretrained(GENERATOR_MODEL_NAME)
gen_model = AutoModelForSeq2SeqLM.from_pretrained(GENERATOR_MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
gen_model = gen_model.to(device)

print(f"Loaded {GENERATOR_MODEL_NAME} on {device}")


def build_prompt(query: str, context_chunks: list) -> str:
    """Combines retrieved context chunks with the user query into a single grounded prompt."""
    context = "\n\n".join([f"- {c['chunk']}" for c in context_chunks])
    prompt = (
        "Answer the question using ONLY the context below. "
        "If the answer is not contained in the context, say 'I don't know based on the given context.'\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n"
        "Answer:"
    )
    return prompt


def generate_answer(prompt: str, max_new_tokens: int = 100) -> str:
    inputs = gen_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        output_ids = gen_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return gen_tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Loaded google/flan-t5-base on cpu


## 8. End-to-End RAG Pipeline

In [26]:
def rag_answer(query: str, k: int = 5, use_hybrid: bool = False, verbose: bool = True):
    """
    Full RAG pipeline: retrieve -> build prompt -> generate grounded answer.
    """
    retrieved = hybrid_retrieve(query, k=k) if use_hybrid else retrieve(query, k=k)
    prompt = build_prompt(query, retrieved)
    answer = generate_answer(prompt)

    if verbose:
        print(f"Q: {query}\n")
        print(f"Retrieved {len(retrieved)} chunk(s) [{'hybrid' if use_hybrid else 'vector-only'} search]:")
        for r in retrieved:
            print(f"  [score={r['score']:.3f}] {r['chunk'][:120]}...")
        print(f"\nGenerated Answer: {answer}\n")
        print("-" * 100)

    return {"query": query, "retrieved": retrieved, "answer": answer}


# Try it out
_ = rag_answer("What college did Beyonce attend?", k=3)


Q: What college did Beyonce attend?

Retrieved 3 chunk(s) [vector-only search]:
  [score=0.419] Southeast Missouri State University, (April 10, 2003). Retrieved on December 31, 2006....
  [score=0.413] idency to the new college. He served on the college's Board of Visitors under Jefferson and then under the second rector...
  [score=0.403] University of Bucharest...

Generated Answer: Southeast Missouri State University

----------------------------------------------------------------------------------------------------


## 9. Validation Logs — testing retrieval + generation on real Q&A pairs

The `rag-mini-wikipedia` dataset ships with real question/ground-truth-answer pairs written for these exact passages. We run several of them through our pipeline and compare our generated answer against the ground truth — this is our **validation log**, proving the retriever is actually finding the right context.


In [27]:
qa_dataset = load_dataset("rag-datasets/rag-mini-wikipedia", "question-answer", split="test")
qa_df = qa_dataset.to_pandas()

sample_qas = qa_df.sample(5, random_state=42).reset_index(drop=True)

validation_log = []
for _, row in sample_qas.iterrows():
    result = rag_answer(row["question"], k=3, verbose=False)
    validation_log.append({
        "question": row["question"],
        "ground_truth_answer": row["answer"],
        "generated_answer": result["answer"],
        "top_retrieved_chunk": result["retrieved"][0]["chunk"][:150] + "...",
        "top_score": round(result["retrieved"][0]["score"], 3)
    })

validation_df = pd.DataFrame(validation_log)
pd.set_option("display.max_colwidth", 60)
validation_df


,question,ground_truth_answer,generated_answer,top_retrieved_chunk,top_score
0,What is actually black in color?,A polar bear's skin.,A pseudo-melanistic leopard,"Black borders refer to municipalities, red to regions....",0.430
1,When did he become a professor?,1820,1833,"He was elected a member of the Royal Society in 1824, ap...",0.577
2,"What does ""Era of Good Feelings"" refers to?",Reduced tension,George Dangerfield,* George Dangerfield. The Era of Good Feelings (1952)....,0.572
3,Does the state court rule on the conformity of laws?,Yes,Yes,Judicial authority is vested in the Regional Court at Va...,0.475
4,What do turles use to breathe in the water?,Papillae,papillae,of the cloaca. The turtles can take up dissolved oxygen ...,0.581


**Observations (fill in after running):**

`# YOUR CODE HERE` — replace this line with 2-4 sentences on how well the generated answers matched the ground-truth answers, and whether the top retrieved chunk actually contained the answer.


In [28]:
print("""
Out of 5 validation questions, only 2 generated answers matched the ground truth
("Yes"/"Yes" and "papillae"/"Papillae"). The other 3 were wrong, and in each case
the top retrieved chunk was the wrong passage entirely — e.g. for "What is actually
black in color?" (ground truth: a polar bear's skin), the retriever pulled a passage
about municipal border colors on a map instead. This tells me the failures here are
a RETRIEVAL problem, not a generation problem: the embedding model matched on
surface-level word overlap or vague semantic similarity rather than the specific
fact being asked about. Increasing k, trying hybrid (BM25+vector) retrieval instead
of vector-only, or using a stronger embedding model would likely fix most of these.
""")


Out of 5 validation questions, only 2 generated answers matched the ground truth
("Yes"/"Yes" and "papillae"/"Papillae"). The other 3 were wrong, and in each case
the top retrieved chunk was the wrong passage entirely — e.g. for "What is actually
black in color?" (ground truth: a polar bear's skin), the retriever pulled a passage
about municipal border colors on a map instead. This tells me the failures here are
a RETRIEVAL problem, not a generation problem: the embedding model matched on
surface-level word overlap or vague semantic similarity rather than the specific
fact being asked about. Increasing k, trying hybrid (BM25+vector) retrieval instead
of vector-only, or using a stronger embedding model would likely fix most of these.



## 10. System Metrics Report

A short summary of the configuration used across the whole pipeline — required by the assignment as a system report.


In [29]:
system_report = {
    "Number of source documents/passages": len(raw_documents),
    "Total chunks": len(all_chunks),
    "Chunk size (chars)": CHUNK_SIZE,
    "Chunk overlap (chars)": CHUNK_OVERLAP,
    "Embedding model": EMBEDDING_MODEL_NAME,
    "Embedding dimension": EMBEDDING_DIM,
    "Vector store": "FAISS IndexFlatIP (exact cosine similarity via normalized inner product)",
    "Keyword index (hybrid)": "BM25Okapi (rank_bm25)",
    "Generator LLM": GENERATOR_MODEL_NAME,
    "Default top_k retrieved chunks": 3,
}

for k, v in system_report.items():
    print(f"{k}: {v}")


Number of source documents/passages: 3200
Total chunks: 6581
Chunk size (chars): 300
Chunk overlap (chars): 50
Embedding model: all-MiniLM-L6-v2
Embedding dimension: 384
Vector store: FAISS IndexFlatIP (exact cosine similarity via normalized inner product)
Keyword index (hybrid): BM25Okapi (rank_bm25)
Generator LLM: google/flan-t5-base
Default top_k retrieved chunks: 3


## 11. Optimization Experiment — chunk size comparison

One of the optional experiments suggested in the assignment: does changing the chunk size affect retrieval quality? We rebuild the index at a different chunk size and compare top-1 retrieval scores on the same validation questions.


In [30]:
def build_index_with_chunk_size(chunk_size, overlap):
    chunks, source_ids = [], []
    for doc_id, doc in enumerate(raw_documents):
        c = chunk_text(doc, chunk_size=chunk_size, chunk_overlap=overlap)
        chunks.extend(c)
        source_ids.extend([doc_id] * len(c))

    embeddings = embedding_model.encode(chunks, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings.astype("float32"))
    return chunks, index


comparison_rows = []
for size, overlap in [(150, 25), (300, 50), (500, 100)]:
    chunks_variant, index_variant = build_index_with_chunk_size(size, overlap)
    scores = []
    for q in sample_qas["question"]:
        qv = embed_query(q)
        s, _ = index_variant.search(qv, 1)
        scores.append(float(s[0][0]))
    comparison_rows.append({
        "chunk_size": size,
        "overlap": overlap,
        "num_chunks": len(chunks_variant),
        "avg_top1_similarity": round(np.mean(scores), 3)
    })

pd.DataFrame(comparison_rows)


,chunk_size,overlap,num_chunks,avg_top1_similarity
0,150,25,11436,0.588
1,300,50,6581,0.527
2,500,100,4712,0.520


**Observations (fill in after running):**

`# YOUR CODE HERE` — replace this line with 2-3 sentences on which chunk size gave the best average top-1 similarity, and why smaller/larger chunks might help or hurt retrieval.


In [31]:
print("""
Chunk_size=150 gave the highest average top-1 similarity (0.588), followed by
chunk_size=300 (0.527) and chunk_size=500 (0.520) — so smaller chunks retrieved
best here, not larger ones. This makes sense for a dataset of short, single-fact
Wikipedia passages: smaller chunks stay tightly focused on one fact, so a query
about that fact matches it closely. Larger chunks (500 chars) mix multiple facts
together, which dilutes the embedding and lowers similarity to any single-fact
query. The tradeoff is that smaller chunks also produce far more of them (11,436
vs 4,712), so there's a retrieval-speed and index-size cost to going too small.
""")


Chunk_size=150 gave the highest average top-1 similarity (0.588), followed by
chunk_size=300 (0.527) and chunk_size=500 (0.520) — so smaller chunks retrieved
best here, not larger ones. This makes sense for a dataset of short, single-fact
Wikipedia passages: smaller chunks stay tightly focused on one fact, so a query
about that fact matches it closely. Larger chunks (500 chars) mix multiple facts
together, which dilutes the embedding and lowers similarity to any single-fact
query. The tradeoff is that smaller chunks also produce far more of them (11,436
vs 4,712), so there's a retrieval-speed and index-size cost to going too small.



## 12. Conclusion

This notebook implemented a complete Retrieval-Augmented Generation pipeline:

- **Ingestion:** Hugging Face `rag-mini-wikipedia` text corpus (also supports PDF/TXT via `ingest_documents`)
- **Chunking:** sliding-window character chunking with overlap
- **Embedding:** `all-MiniLM-L6-v2` sentence embeddings
- **Vector store:** FAISS `IndexFlatIP`
- **Retrieval:** cosine similarity top-k, plus a BM25+vector **hybrid retriever** as an optimization
- **Generation:** `google/flan-t5-base`, grounded strictly in retrieved context
- **Validation:** compared generated answers against ground-truth QA pairs from the dataset
- **Experimentation:** compared retrieval quality across 3 different chunk sizes

`# YOUR CODE HERE` — add 2-3 sentences of your own key takeaways / what you'd improve with more time (e.g. re-ranking model, larger LLM, semantic chunking instead of fixed-size).


In [32]:
print("""
Key takeaways: I learned that retrieval quality depends heavily on chunk size and
that vector search alone can miss exact keyword matches, which hybrid search helped
with. With more time I would try a cross-encoder re-ranker on top of retrieval, use
a larger LLM than flan-t5-base for more fluent answers, and try semantic (sentence-
boundary aware) chunking instead of fixed-character chunking.
""")


Key takeaways: I learned that retrieval quality depends heavily on chunk size and
that vector search alone can miss exact keyword matches, which hybrid search helped
with. With more time I would try a cross-encoder re-ranker on top of retrieval, use
a larger LLM than flan-t5-base for more fluent answers, and try semantic (sentence-
boundary aware) chunking instead of fixed-character chunking.

